# Brain Tumour Detection (YOLO26m-cls)Notebook-first refactor with real Kaggle dataset download, strict file/label/split verification, YOLO26m-cls training, and honest evaluation outputs.

## Dataset Source and Method ChoiceKaggle dataset: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-datasetTask fit: this dataset provides class-labeled MRI images (not segmentation masks), so classification with YOLO26m-cls is the correct primary method for this project.

In [ ]:
import importlibimport subprocessimport sysdef ensure_package(import_name, pip_name=None):    package_name = pip_name or import_name    try:        importlib.import_module(import_name)    except ImportError:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])ensure_package('kagglehub')ensure_package('numpy')ensure_package('pandas')ensure_package('matplotlib')ensure_package('PIL', 'Pillow')ensure_package('yaml', 'pyyaml')ensure_package('ultralytics')ensure_package('sklearn', 'scikit-learn')print('Dependencies are ready.')

In [ ]:
import jsonimport osimport randomimport shutilfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom PIL import Imagefrom IPython.display import displayfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_scoreSEED = 42random.seed(SEED)np.random.seed(SEED)PROJECT_DIR = Path.cwd()WORK_DIR = PROJECT_DIR / 'working'RAW_DIR = WORK_DIR / 'raw_kaggle'DATA_DIR = WORK_DIR / 'prepared_cls'ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'WORK_DIR.mkdir(parents=True, exist_ok=True)RAW_DIR.mkdir(parents=True, exist_ok=True)DATA_DIR.mkdir(parents=True, exist_ok=True)ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)print(f'Project dir: {PROJECT_DIR}')

## Real Dataset Download

In [ ]:
import kagglehubif not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')dataset_root = Path(kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset'))print(f'Dataset root: {dataset_root}')

## Data Verification: Files, Labels, and Splits

In [ ]:
train_dir = dataset_root / 'Training'test_dir = dataset_root / 'Testing'if not train_dir.exists() or not test_dir.exists():    raise RuntimeError('Expected Training and Testing folders are missing.')train_classes = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])test_classes = sorted([p.name for p in test_dir.iterdir() if p.is_dir()])if len(train_classes) == 0 or len(test_classes) == 0:    raise RuntimeError('No class subfolders found in Training/Testing directories.')if train_classes != test_classes:    raise RuntimeError('Train/Test class names mismatch.')image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}rows = []for split_name, split_dir in [('train', train_dir), ('test', test_dir)]:    for cls in train_classes:        cls_dir = split_dir / cls        imgs = [p for p in cls_dir.iterdir() if p.suffix.lower() in image_exts]        for p in imgs:            rows.append({'split': split_name, 'label': cls, 'image_path': str(p)})df = pd.DataFrame(rows)if len(df) == 0:    raise RuntimeError('No images found after scanning dataset folders.')sample_paths = df['image_path'].sample(n=min(300, len(df)), random_state=SEED).tolist()corrupt = 0for p in sample_paths:    try:        with Image.open(p) as img:            img.verify()    except Exception:        corrupt += 1if corrupt > 0:    raise RuntimeError(f'Corrupted images found in sample check: {corrupt}')split_counts = df.groupby(['split', 'label']).size().to_dict()print(f'Total images: {len(df)}')print(f'Classes: {train_classes}')print(f'Split-label counts: {split_counts}')display(df.head(8))

In [ ]:
for split_name in ['train', 'val']:    for cls in train_classes:        (DATA_DIR / split_name / cls).mkdir(parents=True, exist_ok=True)train_df = df[df['split'] == 'train'].copy().reset_index(drop=True)test_df = df[df['split'] == 'test'].copy().reset_index(drop=True)val_parts = []new_train_parts = []for cls in train_classes:    cls_train = train_df[train_df['label'] == cls].copy()    cls_val = cls_train.sample(frac=0.2, random_state=SEED)    cls_new_train = cls_train.drop(cls_val.index)    val_parts.append(cls_val)    new_train_parts.append(cls_new_train)val_df = pd.concat(val_parts, ignore_index=True)train_df = pd.concat(new_train_parts, ignore_index=True)for _, row in train_df.iterrows():    src = Path(row['image_path'])    dst = DATA_DIR / 'train' / row['label'] / src.name    if not dst.exists():        shutil.copy2(src, dst)for _, row in val_df.iterrows():    src = Path(row['image_path'])    dst = DATA_DIR / 'val' / row['label'] / src.name    if not dst.exists():        shutil.copy2(src, dst)prepared_counts = {    'train': len(list((DATA_DIR / 'train').rglob('*'))),    'val': len(list((DATA_DIR / 'val').rglob('*')))}print(f'Prepared counts (including dirs/files): {prepared_counts}')print(f'Train rows: {len(train_df)}, Val rows: {len(val_df)}, Test rows: {len(test_df)}')

In [ ]:
viz = train_df.sample(n=min(9, len(train_df)), random_state=SEED).reset_index(drop=True)fig, axes = plt.subplots(3, 3, figsize=(10, 10))for i in range(len(viz)):    row = viz.iloc[i]    img = Image.open(row['image_path']).convert('RGB')    axes.flatten()[i].imshow(img)    axes.flatten()[i].set_title(row['label'])    axes.flatten()[i].axis('off')for j in range(len(viz), 9):    axes.flatten()[j].axis('off')plt.tight_layout()sample_grid = ARTIFACTS_DIR / 'sample_grid.png'plt.savefig(sample_grid, dpi=140)plt.show()

## YOLO26m-cls Training

In [ ]:
from ultralytics import YOLOweights_candidates = ['yolo26m-cls.pt', 'yolo11m-cls.pt', 'yolov8m-cls.pt']selected_weights = Nonemodel = Nonefor w in weights_candidates:    try:        model = YOLO(w)        selected_weights = w        break    except Exception:        continueif selected_weights is None:    raise RuntimeError('Could not load any cls checkpoint for YOLO.')print(f'Selected weights: {selected_weights}')train_result = model.train(    data=str(DATA_DIR),    epochs=3,    imgsz=160,    batch=32,    project=str(ARTIFACTS_DIR / 'yolo_runs'),    name='brain_tumour_cls',    seed=SEED,    verbose=False)best_model_path = Path(model.trainer.best)print(f'Best model path: {best_model_path}')

## Real Evaluation on Held-Out Test Split

In [ ]:
best_model = YOLO(str(best_model_path))label_to_idx = {name: i for i, name in enumerate(sorted(train_classes))}idx_to_label = {v: k for k, v in label_to_idx.items()}y_true = [label_to_idx[x] for x in test_df['label'].tolist()]y_pred = []y_score = []for p in test_df['image_path'].tolist():    pred = best_model.predict(source=p, verbose=False)[0]    probs = pred.probs.data.cpu().numpy()    pred_idx = int(np.argmax(probs))    y_pred.append(pred_idx)    y_score.append(float(np.max(probs)))test_acc = accuracy_score(y_true, y_pred)test_f1 = f1_score(y_true, y_pred, average='macro')cm = confusion_matrix(y_true, y_pred)print(f'Test accuracy: {test_acc:.4f}')print(f'Test macro F1: {test_f1:.4f}')print('Classification report:')print(classification_report(y_true, y_pred, target_names=[idx_to_label[i] for i in sorted(idx_to_label.keys())]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))im = ax.imshow(cm, cmap='Blues')ax.set_title('Test Confusion Matrix')ax.set_xlabel('Predicted')ax.set_ylabel('True')ticks = list(range(len(train_classes)))ax.set_xticks(ticks)ax.set_yticks(ticks)ax.set_xticklabels([idx_to_label[i] for i in ticks], rotation=45, ha='right')ax.set_yticklabels([idx_to_label[i] for i in ticks])for i in range(cm.shape[0]):    for j in range(cm.shape[1]):        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')plt.tight_layout()cm_png = ARTIFACTS_DIR / 'confusion_matrix.png'plt.savefig(cm_png, dpi=140)plt.show()

## Honest Qualitative Analysis

In [ ]:
pred_df = test_df.copy().reset_index(drop=True)pred_df['y_true'] = y_truepred_df['y_pred'] = y_predpred_df['score'] = y_scorepred_df['correct'] = (pred_df['y_true'] == pred_df['y_pred']).astype(int)hard = pred_df[pred_df['correct'] == 0].head(6)if len(hard) == 0:    hard = pred_df.sample(n=min(6, len(pred_df)), random_state=SEED)fig, axes = plt.subplots(2, 3, figsize=(12, 8))for i in range(len(hard)):    row = hard.iloc[i]    img = Image.open(row['image_path']).convert('RGB')    true_lbl = idx_to_label[int(row['y_true'])]    pred_lbl = idx_to_label[int(row['y_pred'])]    axes.flatten()[i].imshow(img)    axes.flatten()[i].set_title(f't={true_lbl} p={pred_lbl} s={row["score"]:.2f}')    axes.flatten()[i].axis('off')for j in range(len(hard), 6):    axes.flatten()[j].axis('off')plt.tight_layout()qual_png = ARTIFACTS_DIR / 'qualitative_cases.png'plt.savefig(qual_png, dpi=140)plt.show()

## Save Real Outputs

In [ ]:
pred_csv = ARTIFACTS_DIR / 'test_predictions.csv'pred_df.to_csv(pred_csv, index=False)metrics = {    'dataset': 'masoudnickparvar/brain-tumor-mri-dataset',    'classes': sorted(train_classes),    'num_total_images': int(len(df)),    'num_train_images': int(len(train_df)),    'num_val_images': int(len(val_df)),    'num_test_images': int(len(test_df)),    'selected_weights': selected_weights,    'best_model_path': str(best_model_path),    'test_accuracy': float(test_acc),    'test_macro_f1': float(test_f1)}metrics_json = ARTIFACTS_DIR / 'metrics.json'with open(metrics_json, 'w', encoding='utf-8') as f:    json.dump(metrics, f, indent=2)manifest = {    'dataset_url': 'https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset',    'dataset_root': str(dataset_root),    'prepared_data_dir': str(DATA_DIR),    'metrics_json': str(metrics_json),    'predictions_csv': str(pred_csv),    'sample_grid_png': str(sample_grid),    'confusion_matrix_png': str(cm_png),    'qualitative_png': str(qual_png)}manifest_json = ARTIFACTS_DIR / 'artifact_manifest.json'with open(manifest_json, 'w', encoding='utf-8') as f:    json.dump(manifest, f, indent=2)print('Saved outputs:')print(f'- {metrics_json}')print(f'- {pred_csv}')print(f'- {manifest_json}')print(f'- {sample_grid}')print(f'- {cm_png}')print(f'- {qual_png}')

## Limitations and Improvements- This dataset is classification-labeled; segmentation metrics are not applicable without mask annotations.- For stronger performance, increase epochs and image size after confirming hardware limits.- For medical robustness, add cross-dataset validation and calibration analysis before deployment.